# Improved Models with Scaling, Class Balance, and Tuning

In [4]:

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

## Define model pipeline (Scaler + SVM)

In [2]:

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(class_weight="balanced"))
])


## Hyperparameter tuning

In [5]:
OUTPUT_PATH = 'processed_data'
# ── CARICA DATI PREPROCESSATI ────────────────────────────────────────
X_visual  = np.load(f'{OUTPUT_PATH}/X_visual_pca.npy')   # (N, 64)
X_audio   = np.load(f'{OUTPUT_PATH}/X_audio.npy')        # (N, 128)
y_encoded = np.load(f'{OUTPUT_PATH}/y_encoded.npy')      # (N,)
classes   = np.load(f'{OUTPUT_PATH}/label_classes.npy', allow_pickle=True)

# Feature concatenate per i modelli ML classici
X_combined = np.hstack([X_visual, X_audio])  # (N, 192)


param_grid = {
    "svm__C": [0.1, 1, 10],
    "svm__gamma": ["scale", 0.01, 0.001],
    "svm__kernel": ["rbf"]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(pipeline, param_grid, cv=cv, scoring="f1_macro", n_jobs=-1)


## Fit model (replace X_train, y_train)

In [6]:

grid.fit(X_combined, y_encoded)

print("Best params:", grid.best_params_)
print("Best F1:", grid.best_score_)


Best params: {'svm__C': 1, 'svm__gamma': 0.01, 'svm__kernel': 'rbf'}
Best F1: 0.4417670626311253


## Evaluation (replace X_test, y_test)

In [8]:

y_pred = grid.predict(X_combined)

print(classification_report(y_encoded, y_pred))
print(confusion_matrix(y_encoded, y_pred))


              precision    recall  f1-score   support

           0       0.95      1.00      0.97        52
           1       0.92      0.98      0.95       136
           2       1.00      0.95      0.97       258

    accuracy                           0.96       446
   macro avg       0.95      0.98      0.96       446
weighted avg       0.97      0.96      0.96       446

[[ 52   0   0]
 [  2 133   1]
 [  1  12 245]]
